# Phase 1: Problem Definition  
## Dynamic Asset Allocation for Retirement

## 1. Problem Specialization and Practical Relevance

This project studies a retirement-oriented personal finance problem. We consider an investor who is currently 30 years old and plans to retire at age 65. Each year, the investor decides how to allocate wealth across a set of investment strategies in order to grow wealth over time while managing risk.

This problem is practically relevant because retirement investing is inherently a sequential decision problem under uncertainty. Future wealth depends not only on current market conditions, but also on a long sequence of allocation decisions, realized returns, and the investor's evolving tolerance for risk. This makes the problem a natural fit for stochastic control and, in particular, a Markov Decision Process (MDP).


---

## 2. Project Objective

The objective of this project is to determine an optimal dynamic asset-allocation policy from age 30 to age 65.

At each decision time, the investor observes the current state of their financial situation and chooses an allocation across investment strategies. The goal is to maximize expected utility of retirement wealth.

We model this as a finite-horizon MDP with horizon

$T = 65 - 30 = 35$

years.

---

## 3. Three Incrementally Diluted Versions of the Problem

### 3.1 Ideal Practical Version

This is the version we would ideally like to solve if we had ample time and resources.

The state includes:
- current age,
- current wealth,
- annual labor income,
- expected future contributions,
- current market regime,
- inflation regime,
- tax status,
- retirement account balances,
- current portfolio composition.

The action includes:
- allocation across multiple asset classes such as cash, bonds, domestic equities, international equities, and alternatives,
- contribution decisions,
- withdrawal or consumption decisions,
- rebalancing decisions subject to taxes and transaction costs.

The objective is to maximize expected lifetime utility, potentially including both pre-retirement consumption and post-retirement consumption, subject to realistic constraints such as:
- no short-selling,
- leverage limits,
- transaction costs,
- taxes,
- retirement account rules,
- minimum liquidity requirements.

This would be the most practically useful version and could serve as the foundation for a commercial retirement-planning tool.

### 3.2 Realistic Course Version (Phase 3)

This is the richer version solved with deep reinforcement learning (DQN).

**State:** $(t, W_t, \text{regime}_t)$ — time step, continuous wealth, and current market regime. The state is encoded as a 5-dimensional normalized vector:

$$\mathbf{s}_t = \left[\frac{t}{35},\; \frac{\log_{10} W_t}{\log_{10} W_{\max}},\; \mathbf{1}_{\text{bull}},\; \mathbf{1}_{\text{neutral}},\; \mathbf{1}_{\text{bear}}\right]$$

The log-wealth normalization is natural because wealth grows roughly exponentially. One-hot regime encoding avoids imposing an arbitrary numeric ordering on the regimes.

**Market regimes:** three Markov regimes — bull, neutral, and bear — calibrated from 1990–2024 Yahoo Finance data (17 bull years, 13 neutral, 5 bear). Returns within each regime are drawn from regime-dependent normal distributions (e.g., VFINX: $+26.3\%$ in bull, $+5.9\%$ in neutral, $-19.7\%$ in bear). The empirical regime transition matrix is estimated with Laplace smoothing.

**Action:** six discrete strategies (Scheme B) — the three pure fund picks (VFSTX, VBMFX, VFINX) plus three equal-weight 50/50 blends (VFSTX+VBMFX, VBMFX+VFINX, VFSTX+VFINX). This scheme was selected via sensitivity analysis over three action-space designs.

**Switching costs:** if the agent changes its investment choice from one year to the next, a transaction cost of 0.5% of current wealth is deducted, encouraging strategic consistency.

**Solution:** a Deep Q-Network (DQN) with a 5-input, two-hidden-layer MLP (64 units each, ReLU activations, 4,934 total parameters). Training uses a replay buffer (capacity 50,000), a periodically-updated target network, and $\varepsilon$-greedy exploration decaying from 1.0 to 0.05 over approximately 590 episodes.

**Additional experiments:** a sensitivity analysis over three action-space granularities and an ablation study over five reward-function designs (log-wealth shaping vs. various CRRA-based step rewards) are conducted to characterize robustness and reward-design tradeoffs.

### 3.3 Simplified Phase 2 Version

This is the smaller version solved using exact Dynamic Programming (backward induction).

**State:** $(t, b_i)$ — time step $t \in \{0,\dots,35\}$ (equivalently age 30–65) and wealth bin index $b_i$ drawn from a log-spaced grid of 60 bins spanning \$1,000–\$5,000,000.

**Action:** discrete choice among three real index funds:
- **VFSTX** — Vanguard Short-Term Investment-Grade (cash/savings proxy),
- **VBMFX** — Vanguard Total Bond Market Index (bonds),
- **VFINX** — Vanguard 500 Index Fund (broad US equity).

**Return model:** each fund's annual return is drawn from a three-scenario (bear/neutral/bull) discrete distribution fit to 1990–2024 empirical data from Yahoo Finance. Calibrated scenarios for VFINX, for example, are approximately −10.1% (bear), +11.7% (neutral), and +28.4% (bull).

**Wealth transition:** $W_{t+1} = (W_t + c)(1 + r_t)$, where the annual contribution is $c = \$10{,}000$ and $r_t$ is drawn from the selected fund's scenario distribution. The result is clipped and snapped to the nearest wealth grid point.

**Reward:** a log-wealth shaping reward $r_t = \alpha \cdot \Delta\!\log W_t$ with $\alpha = 0.01$ is applied at every step. This matches the Phase 3 training objective and provides a dense learning signal; no sparse terminal CRRA utility is used.

**Solution:** exact backward induction over the $35 \times 60 = 2{,}100$-state space. The Bellman update requires $3 \times 3 = 9$ expected-value computations per state and completes in milliseconds, yielding a globally optimal policy $\pi^*(t, b_i)$.

---

## 4. MDP Formulation

We define an MDP through states, actions, rewards, transitions, and a policy that maps states to actions. Our retirement asset-allocation problem fits that framework naturally.

### 4.1 Time

We use discrete annual time steps:

$t = 0, 1, 2, \dots, T$

where $t=0$ corresponds to age 30 and $T=35$ corresponds to age 65.

Let age evolve deterministically as:

$A_t = 30 + t$

---

### 4.2 State Space

For the main project version, define the state at time $t$ as

$S_t = (A_t, W_t, M_t)$

where:
- $A_t$ is current age,
- $W_t$ is current wealth,
- $M_t$ is the market regime.

Optionally, for a richer formulation, we may include the current allocation:

$S_t = (A_t, W_t, M_t, x_t)$

The Markov property is satisfied if the state contains all information needed so that the distribution of next wealth and reward depends only on the current state and action.

---

### 4.3 Action Space

At each year $t$, the investor chooses an allocation across investment strategies.

#### Discrete-action version

A simple version is

$a_t \in \{\text{Conservative}, \text{Balanced}, \text{Aggressive}\}$

#### Continuous-action version

A richer version uses portfolio weights

$a_t = (\omega_t^{(1)}, \omega_t^{(2)}, \dots, \omega_t^{(n)})$

subject to

$\sum_{i=1}^n \omega_t^{(i)} = 1, \qquad \omega_t^{(i)} \ge 0$

where $\omega_t^{(i)}$ is the fraction of wealth allocated to asset or strategy $i$.

---

### 4.4 State Transitions

Let $R_{t+1}(a_t, M_t, \varepsilon_{t+1})$ denote the random portfolio return from time $t$ to $t+1$, where $\varepsilon_{t+1}$ is a random shock.

If $C_t$ denotes the external contribution made at time $t$, then wealth evolves according to

$W_{t+1} = (W_t + C_t)\bigl(1 + R_{t+1}(a_t, M_t, \varepsilon_{t+1})\bigr)$

Age evolves as

$A_{t+1} = A_t + 1$

If market regimes are included, they follow a Markov transition law of the form

$\mathbb{P}(M_{t+1} = m' \mid M_t = m)$

This allows the environment to capture changing return conditions over time.

---

## 5. Reward Function and Utility

We use utility of terminal retirement wealth as the main reward.

For intermediate years,

$R_t = 0 \qquad \text{for } t = 0,1,\dots,T-1$

At retirement,

$R_T = U(W_T)$

A natural utility choice is Constant Relative Risk Aversion (CRRA) utility:

$U(w;\gamma) =
\begin{cases}
\frac{w^{1-\gamma}-1}{1-\gamma}, & \gamma \neq 1 \\
\log w, & \gamma = 1
\end{cases}$

Here, $\gamma$ is the coefficient of relative risk aversion.

---

## 6. Objective Function

The objective is to find a policy $\pi$ that maps states to actions and maximizes expected total reward:

$\max_{\pi} \mathbb{E}^{\pi}\left[\sum_{t=0}^{T} R_t\right]$

Since all intermediate rewards are zero, this reduces to

$\max_{\pi} \mathbb{E}^{\pi}[U(W_T;\gamma)]$

Equivalently, the optimal value function is

$V_t(s) = \max_{\pi} \mathbb{E}^{\pi}\left[\sum_{\tau=t}^{T} R_\tau \mid S_t = s\right]$

and the Bellman optimality equation is

$V_t(s) = \max_{a \in \mathcal{A}(s)} \mathbb{E}\left[ R_t + V_{t+1}(S_{t+1}) \mid S_t = s, A_t = a \right]$

with terminal condition

$V_T(s) = U(W_T;\gamma)$

---

## 7. Simplified Strategy Set for Implementation

To keep the action space manageable, we begin with three investment strategies:

- **Conservative**: high bond/cash weight, low equity weight
- **Balanced**: moderate bond weight and moderate equity weight
- **Aggressive**: high equity weight, low bond/cash weight

These strategies provide an interpretable action space.

---

## 8. Practical Nuances and Modeling Choices

Several practical aspects motivate this formulation:

1. Retirement investing is a sequential problem, not a one-shot optimization.
2. The same allocation may be appropriate at age 30 but inappropriate at age 60.
3. The investor's objective is not simply to maximize expected wealth, but to maximize risk-adjusted utility.
4. A realistic model quickly becomes high-dimensional, which motivates solving a simplified DP version first and a richer RL version later.

In the full practical version, we may later add:
- transaction costs,
- taxes,
- stochastic income,
- inflation,
- more asset classes,
- contribution rules,
- post-retirement withdrawals.

---

## 9. Code Specification as a MarkovDecisionProcess Class

The project guidance asks that the problem also be specified in code as a `MarkovDecisionProcess` class. At a high level, the class should support the following components:

- representation of states,
- action generation from a state,
- state-transition sampling or probabilities,
- reward computation,
- terminal-state identification.

A high-level interface would include methods such as:

- `actions(state)`
- `step(state, action)`
- `is_terminal(state)`

The key design principle is to parameterize the same environment so that:
- a small parameter setting can be solved with DP/ADP in Phase 2,
- a richer parameter setting can be solved with RL in Phase 3.

---

## 10. Expected Phase 2 and Phase 3 Progression

### Phase 2
We will solve a simplified version with:
- discrete wealth bins,
- discrete strategy choices,
- simple return distributions,
- finite horizon.

This should allow us to use backward induction or value iteration.

### Phase 3
We will extend the same MDP to a more realistic version with:
- richer market dynamics,
- larger state space,
- possibly continuous actions,
- simulation-based training using RL.

This progression matches the intended structure of the course project.

---

## 11. Summary

This project formulates retirement investing as a finite-horizon MDP in which a 30-year-old investor dynamically allocates wealth until retirement at age 65. The project begins with a simplified discrete-action model suitable for DP and then extends to a richer simulation environment suitable for RL. The goal is to learn a dynamic allocation policy that maximizes expected utility of terminal retirement wealth.